#🧠 **Redes Neuronales Convolucionales (CNN)**

---

###🌀 ***¿Qué es una Red Neuronal?***
Una ***red neuronal*** es un programa inspirado en la estructura y el funcionamiento del cerebro humano. Su objetivo principal es aprender a detectar patrones complejos en los datos. Un ***red neuronal*** se entrena con ejemplos y ella aprende las reglas por sí misma.

---

###⚠️ ***Problema a Resolver***
El problema planteado surge de la necesidad de desarrollar un sistema basado en ***redes neuronales*** que permita detectar si un trabajador de la construcción utiliza o no casco de seguridad a partir de imágenes capturadas en obras. La motivación surge de la necesidad de mejorar las condiciones de seguridad laboral y reducir accidentes, ya que el uso del casco es un requisito obligatorio que muchas veces resulta difícil de controlar de forma manual.

- El objetivo es aumentar la eficiencia en la supervisión y contribuir a la prevención de riesgo laboral mediante la inteligencia artificial.

---

###🏆 ***Red Neuronal Convolucional***
Las ***redes neuronales convolucionales*** son las arquitecturas más adecuadas para el procesamiento de imágenes, ya que sus capas ***Conv2D*** pueden identificar patrones espaciales (formas, bordes, texturas) relevantes, como la presencia de un casco en la cabeza del trabajador. El uso de ***MaxPooling2D*** permite además que el modelo sea más robusto a la posición del objeto en la imagen.

---

###👥 ***Colaboradores***
- Martin: https://github.com/bentadev
- Maria: https://github.com/yanin-dev
- Rosmery: https://github.com/ros-arango
- Micaela: https://github.com/MICAELA-IA
- Emmanuel:

##🧰 **1. Preparación y Exploración del Dataset desde Kaggle en Colab**
Vamos a preparar el entorno de ***Google Colab*** para trabajar con un ***dataset*** de Kaggle. Esto incluye instalar librerías necesarias, autenticar nuestra cuenta de Kaggle, descargar y descomprimir el ***dataset***, y explorar la estructura de carpetas.
- 1.1 Instalación y carga de credenciales de Kaggle
- 1.2 Configuración de las credenciales en el entorno.
- 1.3 Descarga y descompresión del dataset.
- 1.4 Exploración de la estructura de carpetas.

In [ ]:
import os
from getpass import getpass

# Pedimos las credenciales de Kaggle al usuario.
# getpass oculta la clave mientras se escribe.
print("Por favor, ingresa tus credenciales de la API de Kaggle:")
kaggle_username = input("Ingresa tu KAGGLE_USERNAME: ")
kaggle_key = getpass("Ingresa tu KAGGLE_KEY: ")

# Configuramos las variables de entorno.
os.environ['KAGGLE_USERNAME'] = kaggle_username
os.environ['KAGGLE_KEY'] = kaggle_key

print("\n¡Credenciales de Kaggle configuradas exitosamente!")

In [ ]:
# Se descarga el dataset desde Kaggle.
!kaggle datasets download -d lbquctrung/worksite-safety-monitoring-dataset
print("El dataset ha sido descargado. Ahora se está descomprimiendo...")

# Se descomprime el dataset descargado.
!unzip -qo worksite-safety-monitoring-dataset.zip
print("¡El dataset se ha descomprimido correctamente!")

In [ ]:
# Se intala 'tree', que sirve para ver la estructura de carpetas y subcarpetas.
!apt-get install -qq tree

# Se usa el comando 'tree' para mostrar solo las carpetas del dataset.
!tree -d Worksite-Safety-Monitoring-Dataset/

##⚡ **2. Preparación de TensorFlow y Carga de los Datasets**
Vamos a preparar el entorno de trabajo con ***TensorFlow*** y ***Keras*** para entrenar nuestra ***red neuronal convolucional***. Esto incluye importar las librerias necesarias, definir parámetros como tamaño de imagen y batch size, cargar los ***datasets*** de entrenamiento y validación, optimizar su rendimiento y visualizar algunas imágenes para entender mejor la distribución de clases.
- 2.1 Importación de librerías
- 2.2 Definición de parámetros
- 2.3 Carga de datasets
- 2.4 Clases encontradas en el dataset
- 2.5 Optimización de datasets
- 2.6 Visualiación de imágenes de ejemplo

In [ ]:
import tensorflow as tf # TensorFlow es la librería base para crear y entrenar redes neuronales.
from tensorflow import keras # Keras es una API de alto nivel que facilita construir, entrenar y evaluar redes neuronales.
from tensorflow.keras import layers # Layers contiene los tipos de capas para utilizar en la red neuronal (convolucionales, densas, pooling, etc).
from tensorflow.keras.models import Sequential # Sequential nos permite crear un modelo apilando las capas de manera lineal (ideal para una red básica).

In [ ]:
# Se define el tamaño (en píxeles) al que vamos a redimensionar todas las imágenes antes de pasarlas a la red neuronal.
IMG_HEIGHT = 128
IMG_WIDTH = 128
IMG_SIZE = (IMG_HEIGHT, IMG_WIDTH)

# Se define el número de imágenes que se procesarán juntas en cada paso del entrenamiento. Trabajar por lotes es más eficiente que procesarlas juntas.
BATCH_SIZE = 32

# Se almacenan en las variables las rutas de las carpetas entrenamietno y validación.
entrenamiento_dir = 'Worksite-Safety-Monitoring-Dataset/train'
validacion_dir = 'Worksite-Safety-Monitoring-Dataset/valid'

In [ ]:
# Se crea un dataset de entrenamiento a partir de las imagenes de entrenamiento_dir.
entrenamiento_ds = tf.keras.utils.image_dataset_from_directory(
  entrenamiento_dir,
  seed=123, # Es una semila para que la división y el orden de las imágenes sea reproducible.
  image_size=IMG_SIZE, # Se redimensionan todas las imágenes a 128x128 píxeles.
  batch_size=BATCH_SIZE # Se organiza las impagenes en lotes de 32 para entrenar de manera eficiente.
)

# Se crea un dataset de validacion a partir de las imagenes de validacion_dir.
validacion_ds = tf.keras.utils.image_dataset_from_directory(
  validacion_dir,
  seed=123,
  image_size=IMG_SIZE,
  batch_size=BATCH_SIZE
)

In [ ]:
# Se obtiene los nombres de las clases que TensorFlow detectó al leer la carpeta entrenamiento_ds.
# Esto nos confirma que TensorFlow leyó correctamente la estructura de carpetas y reconoció las categorías del dataset.
nombre_clases = entrenamiento_ds.class_names
print("Las clases encontradas en el dataset son:", nombre_clases)

In [ ]:
# Se define la constante AUTOTUNE para que TensorFlow decida automáticamente el número óptimo de hilos para cargar datos en paralelo.
# Esto ayuda a mejorar el rendimiento durante el entrenamiento.
AUTOTUNE = tf.data.AUTOTUNE

# Se optimiza el dataset de entrenamiento.
# cache() guarda en memoria los datos para que no tenga que leerlos del disco en cada época.
# shuffle(1000) mezcla las imágens aleatoriamente para que el modelo no vea siempre el mismo orden.
# prefetch(buffer_size=AUTOTUNE) prepara los lotes de datos mientras la GPU/CPU está entrenando, reduciendo tiempos de espera.
entrenamiento_ds = entrenamiento_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)

# Se optimiza el dataset de validacion.
validacion_ds = validacion_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("¡Los datasets fueron optimizados!")

In [ ]:
import matplotlib.pyplot as plt # Matplotlib para crear gráficos y mostrar imágenes en el notebook.

# Se crea una figura de 10x10 pulgadas, que será el contenedor donde se mostrarán las imágenes.
plt.figure(figsize=(10, 10))

# Se toma el primer lote (batch) del dataset entrenamietno_ds.
# images contiene las imágenes del batch.
# labels contiene las etiquetas correspondientes.
for images, labels in entrenamiento_ds.take(1):

  # Se crea una rejilla 3x3 subplots (cuadrícula de subgráficos) para mostrar las 9 primeras imágenes del batch.
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(nombre_clases[labels[i]])
    plt.axis("off")

##🤖 **3. Construcción y Entrenamiento del Modelo (CNN)**
En esta sección se contruye, configura y entrena a la ***Red Neuronal Convolucional*** (CNN) capaz de clasificar imágenes del ***dataset***. El objetivo es que la red aprenda a distinguir entre imágenes seguras (casco) y no seguras (no casco).
- 3.1 Definición del modelo
- 3.2 Compilación del modelo
- 3.3 Entrenamiento del modelo
- 4.4 Visualización del entrenamiento

In [ ]:
# Se crea un modelo secuencial. Las capas se apilan una tras otras, desde la entrada hasta la salida.
# Este modelo es ideal apila capas de manera lineal, es fácil de entender y suficiente para tareas de clasificación de imágenes.
model = Sequential([

  # Se normalizan las imágenes, diviendo cada píxel por 255 para que los valores estén entre 0 y 1.
  # Esta normalización ayuda a que la red aprenda más rápido, sea más estable y produzca mejores resultados.
  # Se indica el tamaño de la imagen (alto x ancho) y 3 canales de color (RGB)
  layers.Rescaling(1./255, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),

  # primera capa convolucional con 16 filtros de tamaño 3x3 (píxeles).
  # Esta capa detecta patrones simples como bordes, texturas o esquinas.
  # padding='same' mantiene el tamaño de la imagen.
  # activation='relu' aplica la función ReLU para introducir no linealidad, ayudando al modelo a aprender patrones complejos.
  layers.Conv2D(16, (3, 3), padding='same', activation='relu'),

  # Capa de pooling que reduce el tamaño espacial de las imágenes (alto y ancho), conservando las características más importantes.
  # Esto reduce la dimensionalidad y acelera el entrenamiento.
  layers.MaxPooling2D(),

  # Segunda capa convolucional, ahora con 32 filtros.
  # Detecta y aprende patrones más complejos combinado la información extraída por la primera capa.
  layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
  layers.MaxPooling2D(),

  # Tercera capa convolucional, ahora con 64 filtros.
  # Detecta patrones aún más abstractos y complejos, combinando las características de las capas anteriores.
  layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
  layers.MaxPooling2D(),

  # Capa de 'aplanado' (Flatten).
  # Las capas convolucionales producen 'mapas' de características en 2D.
  # Esta capa los 'aplana' y los convierte en un vector largo (lista 1D).
  # Esto permite que la información de todas las activaciones de las capas convolucionales se pueda pasar a una capa Dense (cerebro de la red).
  layers.Flatten(),

  # Capa totalmente conectada (Dense) con 128 neuronas.
  # Toma el vector 1D y combina todas las características extraídas por las capas convolucionales para aprender a clasificar.
  # 128 neuronas es una cantidad que permite combinar muchas características extraídas de las capas convolucionales y capturar patrones complejos.
  # Más neuronas podrían aprender mejor, pero también aumentan el riesgo de sobreajuste (el modelo solo memoriza en vez de generalizar).
  layers.Dense(128, activation='relu'),

  # Capa de salida para la clasificación binaria.
  # Es una sola neurona porque solo necesitamos clasificar en 'seguro' o 'no seguro' (binario).si
  # La función de activación sigmoid devuelve un valor entre 0 y 1, interpretado como probabilidad de pertenecer a la clase positiva (seguro).
  layers.Dense(1, activation='sigmoid')
])

# Se muestra un resumen completo del modelo y los parámetros.
model.summary()

In [ ]:
# Compilamos el modelo y le damos las intrucciones necesarias.
# optimizer='adam' es un optimizador muy popular que ajusta los pesos de la red durante el entrenamiento (aprende rápido y con buena estabilidad).
# loss='binary_crossentropy' es una función de pérdida que indica qué tan lejos están las predicciones de la realidad (menor pérdida, mayor aprendizaje).
# metrics=['accuracy] es una métrica que se va a mostrar durante el entrenamiento. Accuracy indica el procentaje de predicciones correctas.
model.compile(
  optimizer='adam',
  loss='binary_crossentropy',
  metrics=['accuracy']
)

print("El modelo se compilo correctamente y está listo para entrenarse.")

In [ ]:
# Entrenamos el modelo utilizando el dataset de entrenamiento y validacion.

# Definimos el número de épocas (cuántas veces el modelo va a recorrer todo el dataset de entrenamiento).
# Cada época ajusta los pesos de la red en función a los errores cometidos.
epocas = 10
print(f"Iniciando el entrenamiento con {epocas} épocas...🚀")

# model.fit es la función para entrenar a la red. Devuelve un objeto history que guarda las métricas de entrenamiento y validacion de cada época (loss, accuracy, etc.).
# Le indicamos el dataset de entrenamietno para que aprenda.
# Le indicamos el dataset de validacion para que se evalúe (veremos cómo generaliza).
# Le indicamos cúantas veces pasará por todo el dataset de entrenamiento (épocas).
history = model.fit(
  entrenamiento_ds,
  validation_data=validacion_ds,
  epochs=epocas
)

print("¡Entrenamiento completado!✅")

In [ ]:
# Para que los gráficos esten en una fila con dos columnas.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfica para visualizar la precisión (Accuracy).
ax1.plot(history.history['accuracy'], label='Entrenamiento')
ax1.plot(history.history['val_accuracy'], label='Validación')
ax1.set_title('Presición por Época')
ax1.set_xlabel('Época')
ax1.set_ylabel('Precisión')
ax1.legend()

# Gráfica para visualizar la pérdida (Loss).
ax2.plot(history.history['loss'], label='Entrenamiento')
ax2.plot(history.history['val_loss'], label='Validación')
ax2.set_title('Pérdida por Época')
ax2.set_xlabel('Época')
ax2.set_ylabel('Pérdida')
ax2.legend()

plt.show()

##🧠 **4. Modelo Mejorado con Data Augmentation y Dropout (Modelo 2)**
En esta estapa mejoramos nuestra ***red neuronal convolucional*** original aplicando dos técnicas muy comunes para evitar el ***sobreajuste*** y mejorar la capacidad de generalización del modelo: ***Data Augmentation*** y ***Dropout***.
- 4.1 Definición del modelo 2
- 4.2 Compilación del modelo 2
- 4.3 Entrenamiento del modelo 2
- 4.4 Visualización del entrenamiento del modelo 2

---

#####🌱 **Data Augmentation**
Consiste en generar nuevas versiones de las imágenes originales aplicando pequeñas transformaciones aleatorias, como giros, reflejos o zooms. De esta forma, el modelo ve una mayor variedad de ejemplos (imágenes) sin la necesidad de recolectar más datos.

#####❌ **Dropout**
Es una técnica que apaga aleatoriamente algunas neuronas durante el entrenamiento. Esto evita que el modelo dependa demasiado de combinaciones específicas de neuronas y, por lo tanto, reduce el  riesgo de sobreajuste. Estamos forzando al modelo a detectar patrones y que no solo memorice las imágenes.

In [ ]:
# Definimos la capa para el aumento de datos.
# Se crea una secuencia de transformaciones que se aplican solamente durante el entrenamiento.
# Se invierte aleatoriamente la imagen de izquierda a derecha.
# Se indica el tamaño de entrada y los 3 canales RGB.
# Se rota aleatoriamente la imagen hasta un 10% del ángulo total.
# Se acerca o aleja la imagen hasta un 10%.
aumento_datos = Sequential(
  [
    layers.RandomFlip("horizontal", input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
  ],
)

# Definimos un nuevo modelo secuencial, igual que antes pero más robusto (model_v2 para diferenciarlo del anterior).
model_v2 = Sequential([

  # Capa de normalización.
  layers.Rescaling(1./255, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),

  # Capa de aumento de datos,
  aumento_datos,

  # Primer bloque convolucional.
  layers.Conv2D(16, (3, 3), padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Dropout(0.25), # Apaga aleatoriamente un 25% de neuronas durante el entrenamiento, reduciendo el sobreajuste.

  # Segundo bloque convolucional.
  layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Dropout(0.25), # Apaga aleatoriamente un 25% de neuronas durante el entrenamiento, reduciendo el sobreajuste.

  # Tercer bloque convolucional.
  layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Dropout(0.25), # Apaga aleatoriamente un 25% de neuronas durante el entrenamiento, reduciendo el sobreajuste.

  # Capas finales.
  layers.Flatten(),
  layers.Dense(128, activation='relu'),
  layers.Dropout(0.5), # Apaga la mitad de las neuronas, reforzando la generalización.
  layers.Dense(1, activation='sigmoid')
])

# Se muestra un resumen completo del modelo y los parámetros.
model_v2.summary()

In [ ]:
# Compilamos al nuevo modelo (model_v2) y definimos cómo aprenderá durante el entrenamiento.
model_v2.compile(
  optimizer='adam',
  loss='binary_crossentropy',
  metrics=['accuracy']
)

In [ ]:
# Entrenamos al nuevo modelo (model_v2) con el aumento de datos y dropout.

# Definimos la cantida de épocas, en este caso 25. Hay más oportunidad de aprender, pero también más riesgo de sobreajuste.
epocas_v2 = 25

print(f"Iniciando el entrenamiento del modelo v2 con {epocas_v2} épocas...🚀")

# La variable history_v2 almacenará el historial (Accuracy y Loss) del entrenamiento del modelo_v2.
history_v2 = model_v2.fit(
  entrenamiento_ds,
  validation_data=validacion_ds,
  epochs=epocas_v2
)

print("¡Entrenamiento del modelo v2 completado!✅")

In [ ]:
# Para que los gráficos estén en una fila con dos columnas.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfica para visualizar la precisión (Accuracy).
ax1.plot(history_v2.history['accuracy'], label='Entrenamiento')
ax1.plot(history_v2.history['val_accuracy'], label='Validación')
ax1.set_title('Precisión por Época (Modelo v2)')
ax1.set_xlabel('Época')
ax1.set_ylabel('Precisión')
ax1.legend()

# Gráfica para visualizar la pérdida (Loss).
ax2.plot(history_v2.history['loss'], label='Entrenamiento')
ax2.plot(history_v2.history['val_loss'], label='Validación')
ax2.set_title('Pérdida por Época (Modelo v2)')
ax2.set_xlabel('Época')
ax2.set_ylabel('Pérdida')
ax2.legend()

plt.show()

##🧩 **5. Modelo Mejorado con Early Stopping (Modelo 3)**
En esta sección entrenamos nuestro ***modelo mejorado v2*** aplicando ***Early Stopping***. Esta técnica permite detener automáticamente el entrenamiento si la pérdida de validación deja de mejorar, evitando ***sobreajuste*** y ahorrando tiempo de cómputo. El resultado es un modelo más robusto y con pesos que representan la mejor época durante el entrenamiento.
- 5.1 Configuración de Early Stopping
- 5.2 Entrenamiento con Early Stopping
- 5.3 Visualización del entrenamiento del modelo 3

In [ ]:
# Importamos EarlyStopping de Keras, que nos permite detener automáticamente el entrenamiento si la métrica deja de mejorar.
from tensorflow.keras.callbacks import EarlyStopping

# En la variable early_stopping almacena un callback que Keras puede usar durante el entrenamiento para detener el proceso si la red deja de mejorar.
# monitor='val_loss' le indica al callback qué métrica vigilar.
# patience=5 define cuántas épocas sin mejora permite antes de detener el entrenamiento (paciencia).
# restore_best_weights=True indica que al detenerse, el modelo recupera los pesos de la mejor época (la de menor pérdida en validacion).
early_stopping = EarlyStopping(
  monitor='val_loss',
  patience=5,
  restore_best_weights=True
)

In [ ]:
# Entrenamos al nuevo modelo (model_v3) con la parada temprana.

# Definimos la cantidad de épocas, en este caso 50. Recordemos que se dentendrá antes si no hay mejoras.
epocas_v3 = 50

print("Iniciando el entrenamiento del modelo v3 con Early Stopping...🚀")

# Se entrena de nuevo al modelo 2, se guarda el hisorial en history_v3 y se aplica la parada temprana.
history_v3 = model_v2.fit(
  entrenamiento_ds,
  validation_data=validacion_ds,
  epochs=epocas_v3,

  # Se aplica la parada temprana, deteniendo el entrenamiento si val_loss no mejora durante las 5 épocas consecutivas.
  # También recuperará los mejores pesos.
  callbacks=[early_stopping]
)

print("¡Entrenamiento del modelo v3 completado por Early Stopping!✅")

In [ ]:
# Para que los gráficos estén en una fila con dos columnas.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfica para visualizar la precisión (Accuracy).
ax1.plot(history_v3.history['accuracy'], label='Entrenamiento')
ax1.plot(history_v3.history['val_accuracy'], label='Validación')
ax1.set_title('Precisión por Época (Modelo v3)')
ax1.set_xlabel('Época')
ax1.set_ylabel('Precisión')
ax1.legend()

# Gráfica para visualizar la pérdida (Loss).
ax2.plot(history_v3.history['loss'], label='Entrenamiento')
ax2.plot(history_v3.history['val_loss'], label='Validación')
ax2.set_title('Pérdida por Época (Modelo v3)')
ax2.set_xlabel('Época')
ax2.set_ylabel('Pérdida')
ax2.legend()

plt.show()

##🧪 **6. Prueba y Testeo del Modelo**
Una vez completado el ***entrenamiento*** y la ***optimización*** del ***modelo 3***, entramos en la etapa de evaluación. El objetivo de esta sección es medir el rendimiento real de nuestra ***red neuronal convolucional*** con imágenes que nunca ha visto.

- 6.1 Pruebas de cómo clasifica el modelo
- 6.2 Testeo al modelo para obtener su precisión final.

In [ ]:
from tensorflow.keras.utils import load_img, img_to_array # Para cargar la imagen y convertirla en array para el modelo.
from PIL import Image # Para mostrar la imagen original de forma nítida.
import io # Para manejar los datos del archivo subido como bytes.
from google.colab import files # Se importa la librería 'files' para subir o descargar archivos desde la computadora.

# Se le solicita al usuario cargar una imagen.
uploaded = files.upload()

# Se verifica que se haya subido al menos un archivo.
if len(uploaded) == 0:
  print("No se seleccionó ningún archivo.")
else:
  nombre_archivo = list(uploaded.keys())[0]
  datos_archivo = uploaded[nombre_archivo]

  # Prepocesamiento para el modelo.
  img_model = load_img(io.BytesIO(datos_archivo), target_size=(IMG_HEIGHT, IMG_WIDTH))
  img_array = img_to_array(img_model)
  img_batch = tf.expand_dims(img_array, 0)

  # El modelo hace la predicción y se obtiene un score entre 0 y 1 (sigmoid) indicando la probabilidad de que sea 'unsafe'.
  score = model_v2.predict(img_batch)[0][0]

  # Se interpreta el score obtenido (<0.5 es Seguro, >=0.5 es no seguro).
  # También se muestra la confianza en porcentaje.
  print(f"🔀Salida de la función Sigmoid: {score:.4f}")
  if score < 0.5:
    print(f"✅Predicción del modelo: es seguro (con casco).")
    print(f"🛡️El modelo está un {(1 - score) * 100:.2f}% seguro de su predicción.")
  else:
    print(f"❌Predicción del modelo: no es seguro (sin casco).")
    print(f"🛡️El modelo está un {score * 100:.2f}% seguro de su predicción.")

  # Se visualiza la imagen original.
  img_original = Image.open(io.BytesIO(datos_archivo))
  plt.figure(figsize=(6,6))
  plt.imshow(img_original)
  plt.axis('off')
  plt.show()

In [ ]:
# Le hacemos una evaluación final al modelo con la carpeta 'test'.
print("Cargando el dataset de 'test'...⏳")

# En la variable test_dir almacenamos la ruta de la carpeta 'test'.
test_dir = 'Worksite-Safety-Monitoring-Dataset//test'

# Cargamos las imágenes de la carpeta 'test' como un dataset de TensorFlow.
test_ds = tf.keras.utils.image_dataset_from_directory(
  test_dir,
  seed=123,
  image_size=IMG_SIZE,
  batch_size=BATCH_SIZE
)

# Optimizamos la carga de datos usando prefetch, para el el GPU/CPU no espere mientras se cargan las imágenes.
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

print("¡El dataset de 'test' se ha cargado correctamente!✅")
print("Evaluando el modelo final (model_v2) con los datos de 'test'...🔍")


# Hacemos la evaluación del modelo.
resultados = model_v2.evaluate(test_ds)
# history_v3 solo guarda el historial de entrenamiento (loss, accuracy por época).
# El modelo model_v2 ya contiene los pesos finales del entrenamietno con Early Stopping.

print("📝¡Resultados Finales!")
print(f"📉Pérdida (Loss) en el test: {resultados[0]:.4f}")
print(f"🎯Precisión (Accuracy) en el test: {resultados[1] * 100:.2f}%")
print()
print(f"💡Esto significa que el modelo acierta en un {resultados[1] * 100:.0f}% de las predicciones.")

##🧾 **7. Conclusión de la Red Convolucional**
El modelo final cumple con el objetivo demostrativo del trabajo práctico. Se logró construir una ***CNN*** que aprende los patrones principales del uso del casco y, mediante técnicas de regularización (Data Augmentation, Dropout) y optimización (Early Stopping), se resolvió el problema inicial de ***obreajuste***, logrando que el modelo generalice las nuevas imágenes.

Este proyecto sienta las bases para desarrollar un ***sistema de seguridad inteligente en obras***. Con un ***dataset*** más amplio y un entrenamiento más fino, podría convertirse en una herramienta útil para prevenir accidentes laborales y automatizar controles de seguridad.

---

#####💡 **Algunas posibles mejoras:**
El porcentaje de acierto es un gran resultado para una ***Red Neuronal Convolucional*** simple, pero si queremos que el modelo aumente esa precisión deberíamos seguir las siguientes estrategias:
- ***Transfer Learning:*** usar un modelo pre-entrenado cómo MobileNetV2, VGG16 o ResNet50.
- ***Aumentar el Dataset:*** nuestro modelo falló en algunas imágenes de prueba. Esto sugiere que el dataset podría ser más diverso.
- ***Hiperparámetros:*** usar una herramienta como KerasTurner para que pruebe automáticamente cientos de combinaciones.
- ***Aumento de Datos Agresivo:*** podríamos utilizar transformaciones más complejas para hacer el  entrenamiento aún más dificil.